# Dati Willamette per preparare una nuova previsione a un'ora

Quattro finestre del 2024 forniscono 2.559 abbinamenti completi. Questa è una
verifica dei dati: non addestra un modello e non misura nuove prestazioni.
Le fonti sono snapshot USGS archiviati con hash; il quaderno funziona senza rete.
Eseguirlo dalla radice del repository o da `docs/notebooks`.


In [1]:
from pathlib import Path
import sys
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/audit_training_readiness.py').exists())
sys.path.insert(0, str(root / 'scripts'))
import audit_training_readiness as readiness
report = readiness.audit(root)
saved = json.loads((root / 'docs/evidence/willamette-forecast/training-readiness.json').read_text(encoding='utf-8'))
readiness.shared.compare_saved(report, saved)
print('Rapporto riprodotto. Abbinamenti completi:', report['totalAcceptedPairs'])
print('Nuovo modello addestrato:', report['modelFitted'])
print('Nuova accuratezza valutata:', report['forecastSkillEvaluated'])


Rapporto riprodotto. Abbinamenti completi: 2559
Nuovo modello addestrato: False
Nuova accuratezza valutata: False


## Copertura rispetto a tutti gli orari attesi

Non basta contare gli orari con una misura presente: quelli totalmente assenti
devono restare nel denominatore. Per ogni finestra la griglia UTC comprende 660
origini possibili, dopo aver lasciato spazio per la storia e il bersaglio a un'ora.


In [2]:
print('Finestra         Coppie  Origini attese  Copertura  Origini ossigeno assenti')
for w in report['windows']:
    g = w['fullExpectedGrid']
    print(f"{w['window']['id']:<16} {g['acceptedExactGridPairs']:<7} {g['expectedOrigins']:<15} {g['acceptedGridFraction']:<10.2%} {g['missingApprovedOxygenOrigins']}")
autumn = next(w for w in report['windows'] if w['window']['id'] == 'autumn-2024')
print('Autunno: 595 / 660 =', round(100 * autumn['fullExpectedGrid']['acceptedGridFraction'], 2), '%')
print('Le origini presenti erano soltanto', autumn['alignment']['eligibleOrigins'])


Finestra         Coppie  Origini attese  Copertura  Origini ossigeno assenti
winter-2024      660     660             100.00%    0
spring-2024      644     660             97.58%     3
summer-2024      660     660             100.00%    0
autumn-2024      595     660             90.15%     52
Autunno: 595 / 660 = 90.15 %
Le origini presenti erano soltanto 608


## Condizioni osservate e limiti

Gli intervalli sotto descrivono misure approvate, non una legge causale o una
garanzia di rappresentatività. Il filtro pratico anticipato di 500 coppie e 5%
di scarti fra origini presenti viene mantenuto; è distinto dalla copertura
completa della griglia. Nessuna finestra contiene bersagli sotto 5 mg/L, soglia
illustrativa. Eventi critici, disponibilità storica in tempo reale e beneficio
ecologico restano non valutati.


In [3]:
for w in report['windows']:
    o, t = w['ranges']['oxygen'], w['ranges']['temperature']
    print(w['window']['id'], '| ossigeno:', o['min'], '—', o['max'], 'mg/L',
          '| temperatura:', t['min'], '—', t['max'], '°C')
assert not report['modelFitted'] and not report['forecastSkillEvaluated']
assert not report['operationalModelChanged']
assert autumn['fullExpectedGrid']['expectedOrigins'] == 660
assert autumn['fullExpectedGrid']['missingApprovedOxygenOrigins'] == 52
print('Decisione:', report['decision'])


winter-2024 | ossigeno: 12.4 — 14.7 mg/L | temperatura: 5.1 — 8.6 °C
spring-2024 | ossigeno: 11.4 — 12.6 mg/L | temperatura: 9.5 — 12.7 °C
summer-2024 | ossigeno: 7.4 — 9.3 mg/L | temperatura: 20.7 — 25.0 °C
autumn-2024 | ossigeno: 8.8 — 9.9 mg/L | temperatura: 14.8 — 17.8 °C
Decisione: use this audit to plan a separate one-hour experiment; no model promotion


## Percorso concordato

Prima verificare l'utilità della previsione a un'ora; poi studiare le 24 ore e
i tre mesi. Il nuovo protocollo di fit, scelta, calibrazione e verifica deve
usare fasi cronologiche distinte. I valori già ispezionati possono servire alla
preparazione, non essere presentati come una prova finale mai vista.
Il modello operativo, le decisioni civiche e i finanziamenti non cambiano.
